# YOLOv8n 日期區域偵測訓練
**步驟：** 上傳 zip → 解壓 → 訓練 → 下載 best.pt

In [ ]:
# 1. 確認 GPU
!nvidia-smi

In [ ]:
# 2. 安裝 ultralytics
!pip install ultralytics -q

In [ ]:
# 3. 上傳 yolo_dataset_for_colab.zip
from google.colab import files
uploaded = files.upload()  # 選 yolo_dataset_for_colab.zip

In [ ]:
# 4. 解壓
import zipfile
with zipfile.ZipFile('yolo_dataset_for_colab.zip', 'r') as z:
    z.extractall('/content/')
print('解壓完成')
!ls /content/yolo_dataset/

In [ ]:
# 5. 修正 data.yaml 路徑（Colab 路徑不同）
import yaml
with open('/content/yolo_dataset/data.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/yolo_dataset'

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('data.yaml 已更新：')
!cat /content/yolo_dataset/data.yaml

In [ ]:
# 6. 開始訓練
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,          # GPU 可以用更大 batch
    patience=20,
    project='/content/runs',
    name='date_det_v1',
    exist_ok=True,
    device=0,          # GPU
    workers=2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

print(f'Best model: {results.save_dir}/weights/best.pt')

In [ ]:
# 7. 驗證結果
val = model.val(data='/content/yolo_dataset/data.yaml')
print(f'mAP@0.5     : {val.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val.box.map:.4f}')

In [ ]:
# 8. 下載 best.pt
import shutil
shutil.copy('/content/runs/date_det_v1/weights/best.pt', '/content/best.pt')
files.download('/content/best.pt')

In [ ]:
# 9. (選用) 也下載訓練曲線圖
files.download('/content/runs/date_det_v1/results.png')